# 第 13 章：統計檢定決策樹與 A/B Testing 實戰

> **學習目標**
> 1. 掌握統計檢定方法的選擇邏輯（決策樹）
> 2. 理解並運用自動化檢定框架
> 3. 從零到一完成 A/B Testing 全流程
> 4. 學會多重檢定校正以避免假陽性膨脹

| 導覽 | 連結 |
|------|------|
| 上一章 | 12_迴歸分析與模型診斷 |
| 下一章 | 14_進階主題與專案整合 |

---
## Part A — 統計檢定決策樹

### A-1 決策流程圖

選擇統計檢定方法時，依照以下流程逐步判斷：

```
                    ┌─────────────────────┐
                    │   研究問題是什麼？    │
                    └─────────┬───────────┘
                              │
               ┌──────────────┼──────────────┐
               ▼              ▼              ▼
         比較差異         關聯性分析      預測/迴歸
               │              │              │
               ▼              │              ▼
     ┌─────────────────┐      │      線性/邏輯迴歸
     │  依變數類型？    │      │
     └────┬────────┬───┘      │
          │        │          │
        連續     類別          ▼
          │        │    Chi-squared /
          │        │    Fisher exact
          ▼        ▼
    ┌──────────┐  Chi-squared
    │ 幾組？   │
    └──┬───┬───┘
       │   │
     2組  3+組
       │   │
       ▼   ▼
  ┌────────────────────┐
  │  常態性檢定         │
  │  (Shapiro-Wilk)    │
  └────┬──────────┬────┘
   通過(p>0.05) 不通過(p<=0.05)
       │              │
       ▼              ▼
  ┌──────────┐   ┌──────────────┐
  │方差齊性   │   │ 無母數檢定   │
  │(Levene)  │   │ Mann-Whitney │
  └──┬───┬───┘   │ Kruskal-W.  │
   齊  不齊      └──────────────┘
     │   │
     ▼   ▼
  t-test Welch's t   (2組)
  ANOVA  Welch ANOVA (3+組)
```

### A-2 檢定方法速查表

| 場景 | 組數 | 參數檢定 | 無母數替代 | 前提假設 |
|------|------|----------|-----------|----------|
| 單組 vs 已知值 | 1 | 單樣本 t-test | Wilcoxon signed-rank | 常態分布 |
| 兩獨立組 | 2 | 獨立 t-test | Mann-Whitney U | 常態 + 方差齊性 |
| 兩配對組 | 2 | 配對 t-test | Wilcoxon signed-rank | 差值常態 |
| 三組以上獨立 | 3+ | One-way ANOVA | Kruskal-Wallis | 常態 + 方差齊性 |
| 三組以上配對 | 3+ | Repeated ANOVA | Friedman test | 球形性 |
| 類別 vs 類別 | -- | Chi-squared | Fisher exact (小樣本) | 期望次數 >= 5 |
| 相關性 | -- | Pearson r | Spearman rho / Kendall tau | 雙變量常態 |

**選擇原則**：
- 滿足前提假設 → 優先使用參數檢定（統計效力較高）
- 違反假設或小樣本 → 使用無母數替代方法

### A-3 常見錯誤與注意事項

| 常見錯誤 | 正確做法 |
|---------|----------|
| 不檢查常態性直接用 t-test | 先做 Shapiro-Wilk 檢定 |
| 小樣本強行用參數檢定 | n < 30 時優先考慮無母數方法 |
| 忽略方差齊性 | 方差不齊時用 Welch's t |
| 多重比較不校正 | 使用 Bonferroni / BH 校正 |
| 只看 p-value 不看效應量 | 同時報告效應量與信賴區間 |

### A-4 決策函式：自動選擇檢定方法

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


def choose_test(groups, paired=False, alpha=0.05):
    """
    根據資料特性自動選擇統計檢定方法。

    Parameters
    ----------
    groups : list of array-like
        各組資料
    paired : bool
        是否為配對設計
    alpha : float
        顯著水準

    Returns
    -------
    dict : 包含檢定名稱、統計量、p-value、決策理由
    """
    n_groups = len(groups)
    reasons = []

    # Step 1: 常態性檢定
    normality_results = []
    for i, g in enumerate(groups):
        if len(g) >= 3:
            stat, p = stats.shapiro(g)
            is_normal = p > alpha
            normality_results.append(is_normal)
            reasons.append(
                f"Group {i+1} Shapiro p={p:.4f} -> "
                f"{'常態' if is_normal else '非常態'}"
            )
        else:
            normality_results.append(False)
            reasons.append(f"Group {i+1} 樣本數 < 3，無法檢定常態性")

    all_normal = all(normality_results)

    # Step 2: 方差齊性檢定（僅常態時）
    equal_var = True
    if all_normal and n_groups >= 2 and not paired:
        lev_stat, lev_p = stats.levene(*groups)
        equal_var = lev_p > alpha
        reasons.append(
            f"Levene p={lev_p:.4f} -> "
            f"{'方差齊性' if equal_var else '方差不齊'}"
        )

    # Step 3: 選擇檢定
    if n_groups == 2:
        if paired:
            if all_normal:
                test_name = "配對 t-test"
                stat, p = stats.ttest_rel(groups[0], groups[1])
            else:
                test_name = "Wilcoxon signed-rank test"
                stat, p = stats.wilcoxon(groups[0], groups[1])
        else:
            if all_normal:
                if equal_var:
                    test_name = "獨立 t-test (equal var)"
                    stat, p = stats.ttest_ind(
                        groups[0], groups[1], equal_var=True
                    )
                else:
                    test_name = "Welch's t-test (unequal var)"
                    stat, p = stats.ttest_ind(
                        groups[0], groups[1], equal_var=False
                    )
            else:
                test_name = "Mann-Whitney U test"
                stat, p = stats.mannwhitneyu(
                    groups[0], groups[1], alternative='two-sided'
                )
    elif n_groups > 2:
        if all_normal and not paired:
            if equal_var:
                test_name = "One-way ANOVA"
                stat, p = stats.f_oneway(*groups)
            else:
                test_name = "Kruskal-Wallis test (方差不齊)"
                stat, p = stats.kruskal(*groups)
        else:
            test_name = "Kruskal-Wallis test"
            stat, p = stats.kruskal(*groups)
    else:
        test_name = "單樣本 t-test"
        stat, p = stats.ttest_1samp(groups[0], 0)

    reasons.append(f"選擇: {test_name}")
    significant = p < alpha

    return {
        'test': test_name,
        'statistic': stat,
        'p_value': p,
        'significant': significant,
        'reasons': reasons,
    }

In [ ]:
# 示範 1：兩組常態、方差齊性 → 獨立 t-test
np.random.seed(0)
g1 = np.random.normal(100, 15, 50)
g2 = np.random.normal(110, 15, 50)

result = choose_test([g1, g2])
print("=== 決策過程 ===")
for r in result['reasons']:
    print(f"  {r}")
print(f"\n檢定: {result['test']}")
print(f"統計量: {result['statistic']:.4f}")
print(f"p-value: {result['p_value']:.4f}")
print(f"顯著: {'是' if result['significant'] else '否'}")

In [ ]:
# 示範 2：兩組非常態 → Mann-Whitney U
np.random.seed(1)
g3 = np.random.exponential(5, 30)
g4 = np.random.exponential(8, 30)

result2 = choose_test([g3, g4])
print("=== 決策過程 ===")
for r in result2['reasons']:
    print(f"  {r}")
print(f"\n檢定: {result2['test']}")
print(f"p-value: {result2['p_value']:.4f}")

In [ ]:
# 示範 3：三組常態 → ANOVA
np.random.seed(2)
g5 = np.random.normal(50, 10, 40)
g6 = np.random.normal(55, 10, 40)
g7 = np.random.normal(60, 10, 40)

result3 = choose_test([g5, g6, g7])
print("=== 決策過程 ===")
for r in result3['reasons']:
    print(f"  {r}")
print(f"\n檢定: {result3['test']}")
print(f"p-value: {result3['p_value']:.6f}")

In [ ]:
# 示範 4：配對設計 → 配對 t-test
np.random.seed(3)
before = np.random.normal(70, 12, 25)
after = before + np.random.normal(5, 4, 25)  # 處理後平均增加 5

result4 = choose_test([before, after], paired=True)
print("=== 決策過程（配對設計）===")
for r in result4['reasons']:
    print(f"  {r}")
print(f"\n檢定: {result4['test']}")
print(f"p-value: {result4['p_value']:.6f}")

---
## Part B — 自動化檢定框架（UnivariateAnalysisModule）

### B-1 框架設計理念

`UnivariateAnalysisModule` 將統計檢定的五個步驟封裝為一個自動化流程：

1. **常態性檢定** (Shapiro-Wilk)
2. **方差齊性檢定** (Levene's test)
3. **自動選擇檢定方法** (t / Welch / Mann-Whitney / ANOVA / Kruskal-Wallis)
4. **效應量計算** (Cohen's d / rank-biserial)
5. **多重比較校正** (Bonferroni)

搭配 `create_codebook()` 可快速了解資料品質與變數特性。

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 載入工具模組
sys.path.insert(0, '.')
from utils.stat_helpers import UnivariateAnalysisModule, create_codebook

print("工具模組載入成功")

In [ ]:
# 載入 Titanic 資料集
df_titanic = pd.read_csv('datasets/titanic_train.csv', encoding='utf-8-sig')
print(f"資料集大小: {df_titanic.shape}")
print(f"欄位: {list(df_titanic.columns)}")
df_titanic.head()

In [ ]:
# 建立 Codebook
codebook = create_codebook(df_titanic)
codebook

In [ ]:
# 使用 UnivariateAnalysisModule 進行自動化檢定
# 分組變數: Survived (0/1)  →  兩組比較
# 檢定變數: Age, Fare, SibSp, Parch

module = UnivariateAnalysisModule(alpha=0.05)
results = module.run_analysis(
    df=df_titanic,
    group_col='Survived',
    variables=['Age', 'Fare', 'SibSp', 'Parch']
)
results

### B-5 自動化檢定結果解讀

**框架自動完成的步驟：**
1. 對每個數值變數，依 `Survived` 分成兩組
2. 執行 Shapiro-Wilk 常態性檢定
3. 執行 Levene 方差齊性檢定
4. 根據前提假設選擇 t-test / Welch's t / Mann-Whitney U
5. 計算效應量 (Cohen's d 或 rank-biserial)
6. 套用 Bonferroni 多重比較校正

**關鍵欄位說明：**
| 欄位 | 意義 |
|------|------|
| `Test Used` | 框架自動選擇的檢定方法 |
| `p-value` | 原始 p 值 |
| `p-value_corrected` | Bonferroni 校正後 p 值 |
| `Result` | 原始判定（是=顯著） |
| `Result_corrected` | 校正後判定 |
| `Effect Size` | 效應量大小 |

**解讀原則：**
- 以 `Result_corrected` 為最終判斷依據
- 效應量判斷：|d| < 0.2 小、0.2-0.8 中、> 0.8 大

In [ ]:
# 以 Pclass（三組）為分組變數重新分析
module2 = UnivariateAnalysisModule(alpha=0.05)
results2 = module2.run_analysis(
    df=df_titanic,
    group_col='Pclass',
    variables=['Age', 'Fare', 'SibSp', 'Parch']
)
print("=== 以 Pclass 分組的檢定結果（三組 → ANOVA/Kruskal-Wallis）===")
results2

---
## Part C — A/B Testing 基礎概念

### C-1 什麼是 A/B Testing？

**定義**：A/B Testing（又稱隨機對照實驗）是一種將使用者隨機分為兩組（或多組），分別接受不同處理，再透過統計檢定比較結果的實驗方法。

**核心要素：**
- **隨機分配**：消除選擇偏差
- **對照組 (Control)**：維持現狀
- **實驗組 (Treatment)**：接受改變
- **統計檢定**：判斷差異是否顯著

### C-2 常見商業場景

| 產業 | 場景 | 指標 |
|------|------|------|
| 電商 | 新版結帳頁面 | 轉換率、客單價 |
| 社群 | 推薦演算法 | 點擊率、停留時間 |
| SaaS | 定價方案 | 訂閱率、ARPU |
| 遊戲 | 新手引導流程 | 留存率、付費率 |
| 金融 | 風控模型 | 核准率、違約率 |

### C-3 A/B Testing vs 觀察性研究

| 面向 | A/B Testing | 觀察性研究 |
|------|------------|------------|
| 因果推論 | 可建立因果 | 僅能建立相關 |
| 偏差控制 | 隨機分配消除 | 需統計校正 |
| 成本 | 需工程實作 | 使用歷史資料 |
| 倫理 | 需考量實驗倫理 | 較少倫理問題 |
| 適用 | 可介入的場景 | 無法介入的場景 |

### C-4 關鍵統計概念複習

| 概念 | 定義 | A/B Test 中的意義 |
|------|------|-------------------|
| Type I Error (alpha) | 錯誤拒絕 H0 | 宣稱有效但其實無效 |
| Type II Error (beta) | 錯誤接受 H0 | 真的有效但沒偵測到 |
| Power (1-beta) | 正確拒絕 H0 的能力 | 能偵測真實差異的機率 |
| MDE | 最小可偵測效應 | 商業上有意義的最小變化 |
| p-value | 在 H0 為真時觀察到此結果的機率 | 越小越傾向拒絕 H0 |

---
## Part D — A/B Testing 九步流程

### 完整九步流程

```
Step 1  定義商業問題與假設
   │    H0: 新版頁面的客單價 = 舊版
   │    H1: 新版頁面的客單價 ≠ 舊版
   ▼
Step 2  選擇核心指標 (Primary Metric)
   │    主指標: 客單價 (AOV)
   │    護欄指標: 跳出率、頁面載入時間
   ▼
Step 3  計算樣本數 (Power Analysis)
   │    alpha=0.05, power=0.80
   │    最小可偵測效應 (MDE)
   ▼
Step 4  隨機分流設計
   │    使用者層級 vs 請求層級
   │    分流比例 (50/50 or 不等比)
   ▼
Step 5  執行 A/A Test（驗證分流系統）
   │    確認 p > 0.05（兩組無差異）
   ▼
Step 6  正式實驗上線
   │    監控異常、不提前偷看結果
   ▼
Step 7  資料收集與前處理
   │    排除異常值、檢查資料品質
   ▼
Step 8  統計分析
   │    常態性 → 方差齊性 → 選擇檢定
   │    計算效應量、信賴區間
   ▼
Step 9  決策與報告
        效果顯著且實質重要 → 全量上線
        效果不顯著 → 維持現狀或迭代
```

### 各步驟注意事項

| 步驟 | 關鍵注意事項 |
|------|-------------|
| Step 1 | 假設必須在實驗前設定，不可事後調整 |
| Step 2 | 主指標只選一個，護欄指標可以多個 |
| Step 3 | 樣本數不足會降低 power，導致 Type II Error |
| Step 4 | 分流必須保證隨機性，避免 SRM (Sample Ratio Mismatch) |
| Step 5 | A/A Test 失敗 → 必須先修復分流系統 |
| Step 6 | 禁止「偷看」(peeking) — 會膨脹 Type I Error |
| Step 7 | 資料清洗規則應在實驗前定義 |
| Step 8 | 遵循決策樹選擇正確的檢定方法 |
| Step 9 | 統計顯著不等於商業顯著 |

---
## Part E — 電商 A/B Test 實戰

### E-1 情境設定

某電商平台希望測試**新版結帳頁面**是否能提高客單價 (Average Order Value)。

- **對照組 (Control)**：現有結帳頁面
- **實驗組 (Treatment)**：新版結帳頁面（簡化步驟 + 推薦加購）
- **假設**：H0: mu_control = mu_treatment; H1: mu_control != mu_treatment

In [ ]:
# E-2 模擬 A/B Test 資料
np.random.seed(42)

n_control = 200
n_treatment = 200

control = np.random.normal(500, 150, n_control)    # 現有頁面：均值 500, 標準差 150
treatment = np.random.normal(540, 160, n_treatment)  # 新頁面：均值 540, 標準差 160

# 建立 DataFrame
df_ab = pd.DataFrame({
    'group': ['control'] * n_control + ['treatment'] * n_treatment,
    'order_value': np.concatenate([control, treatment])
})

print("=== 描述性統計 ===")
print(f"Control 組:   n={n_control}, mean={control.mean():.2f}, std={control.std():.2f}")
print(f"Treatment 組: n={n_treatment}, mean={treatment.mean():.2f}, std={treatment.std():.2f}")
print(f"觀察到的差異: {treatment.mean() - control.mean():.2f}")

In [ ]:
# E-3 視覺化兩組分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方圖 + KDE
ax1 = axes[0]
sns.histplot(control, kde=True, color='steelblue', label='Control', alpha=0.6, ax=ax1)
sns.histplot(treatment, kde=True, color='coral', label='Treatment', alpha=0.6, ax=ax1)
ax1.axvline(control.mean(), color='steelblue', linestyle='--', linewidth=2, label=f'Control mean={control.mean():.0f}')
ax1.axvline(treatment.mean(), color='coral', linestyle='--', linewidth=2, label=f'Treatment mean={treatment.mean():.0f}')
ax1.set_title('A/B Test: Order Value Distribution')
ax1.set_xlabel('Order Value')
ax1.legend()

# 箱型圖
ax2 = axes[1]
df_ab.boxplot(column='order_value', by='group', ax=ax2)
ax2.set_title('A/B Test: Boxplot Comparison')
ax2.set_xlabel('Group')
ax2.set_ylabel('Order Value')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# E-4 Power Analysis — 事前計算所需樣本數
from statsmodels.stats.power import tt_ind_solve_power

# 預期效應量 (Cohen's d)
expected_diff = 40  # 期望偵測到 40 元的差異
pooled_std = 155    # 預估合併標準差
expected_d = expected_diff / pooled_std

# 計算每組所需樣本數
required_n = tt_ind_solve_power(
    effect_size=expected_d,
    alpha=0.05,
    power=0.80,
    ratio=1.0,         # 兩組等比
    alternative='two-sided'
)

print("=== Power Analysis ===")
print(f"預期效應量 (Cohen's d): {expected_d:.4f}")
print(f"每組所需最小樣本數: {int(np.ceil(required_n))}")
print(f"總計需要: {int(np.ceil(required_n)) * 2} 筆")
print(f"\n目前每組: {n_control} 筆 -> {'足夠' if n_control >= required_n else '不足，建議增加樣本'}")

In [ ]:
# E-5 Power Curve — 不同效應量下所需樣本數
effect_sizes = np.arange(0.1, 1.01, 0.05)
sample_sizes = [
    tt_ind_solve_power(effect_size=es, alpha=0.05, power=0.80, alternative='two-sided')
    for es in effect_sizes
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(effect_sizes, sample_sizes, 'b-o', markersize=4)
ax.axhline(y=200, color='red', linestyle='--', label=f'Current n={n_control}')
ax.axvline(x=expected_d, color='green', linestyle='--', label=f'Expected d={expected_d:.2f}')
ax.set_xlabel("Cohen's d (Effect Size)")
ax.set_ylabel('Required Sample Size per Group')
ax.set_title('Power Curve: Sample Size vs Effect Size (alpha=0.05, power=0.80)')
ax.legend()
ax.set_ylim(0, 1600)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# E-6 常態性檢定 (Shapiro-Wilk)
shapiro_control = stats.shapiro(control)
shapiro_treatment = stats.shapiro(treatment)

print("=== Shapiro-Wilk 常態性檢定 ===")
print(f"Control:   W={shapiro_control.statistic:.4f}, p={shapiro_control.pvalue:.4f}")
print(f"Treatment: W={shapiro_treatment.statistic:.4f}, p={shapiro_treatment.pvalue:.4f}")

alpha = 0.05
control_normal = shapiro_control.pvalue > alpha
treatment_normal = shapiro_treatment.pvalue > alpha
both_normal = control_normal and treatment_normal

print(f"\nControl 常態:   {'是' if control_normal else '否'}")
print(f"Treatment 常態: {'是' if treatment_normal else '否'}")
print(f"兩組皆常態:     {'是 → 可用參數檢定' if both_normal else '否 → 改用無母數檢定'}")

In [ ]:
# E-7 方差齊性檢定 (Levene's test) + 選擇檢定方法
levene_stat, levene_p = stats.levene(control, treatment)

print("=== Levene 方差齊性檢定 ===")
print(f"F={levene_stat:.4f}, p={levene_p:.4f}")
equal_var = levene_p > alpha
print(f"方差齊性: {'是' if equal_var else '否'}")

# 根據前提假設選擇檢定
print("\n=== 選擇並執行檢定 ===")
if both_normal:
    if equal_var:
        t_stat, t_p = stats.ttest_ind(control, treatment, equal_var=True)
        test_name = "獨立 t-test (equal variance)"
    else:
        t_stat, t_p = stats.ttest_ind(control, treatment, equal_var=False)
        test_name = "Welch's t-test (unequal variance)"
else:
    t_stat, t_p = stats.mannwhitneyu(control, treatment, alternative='two-sided')
    test_name = "Mann-Whitney U test"

print(f"檢定方法: {test_name}")
print(f"統計量: {t_stat:.4f}")
print(f"p-value: {t_p:.4f}")
print(f"結論: {'拒絕 H0 — 兩組有顯著差異' if t_p < alpha else '無法拒絕 H0 — 差異不顯著'}")

In [ ]:
# E-8 效應量 — Cohen's d + 95% 信賴區間

def cohens_d(x, y):
    """計算 Cohen's d 效應量（使用合併標準差）。"""
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(
        ((nx - 1) * x.std(ddof=1)**2 + (ny - 1) * y.std(ddof=1)**2)
        / (nx + ny - 2)
    )
    return (x.mean() - y.mean()) / pooled_std


d = cohens_d(treatment, control)

# 效應量解讀
if abs(d) < 0.2:
    effect_label = "小效應 (small)"
elif abs(d) < 0.5:
    effect_label = "小到中效應 (small-medium)"
elif abs(d) < 0.8:
    effect_label = "中效應 (medium)"
else:
    effect_label = "大效應 (large)"

print(f"=== 效應量 ===")
print(f"Cohen's d = {d:.4f} ({effect_label})")
print(f"解讀: Treatment 組比 Control 組高出 {abs(d):.2f} 個合併標準差")

# 95% 信賴區間
diff = treatment.mean() - control.mean()
se_diff = np.sqrt(control.var(ddof=1)/n_control + treatment.var(ddof=1)/n_treatment)
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"\n=== 差異的 95% 信賴區間 ===")
print(f"平均差異: {diff:.2f}")
print(f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"CI 是否包含 0: {'是 → 差異可能為零' if ci_lower <= 0 <= ci_upper else '否 → 差異可信'}")

In [ ]:
# E-9 效應量視覺化
fig, ax = plt.subplots(figsize=(10, 4))

# 差異分布
ax.barh(['Treatment - Control'], [diff], xerr=[[diff - ci_lower], [ci_upper - diff]],
        color='steelblue', capsize=10, height=0.4)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='No difference')
ax.set_xlabel('Difference in Order Value')
ax.set_title(f'A/B Test Result: Difference = {diff:.1f}, 95% CI = [{ci_lower:.1f}, {ci_upper:.1f}]')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### E-10 A/B Test 完整結論報告

**實驗設計：**
- 對照組 (n=200)：現有結帳頁面
- 實驗組 (n=200)：新版結帳頁面
- 主指標：客單價 (Average Order Value)

**統計分析結果：**
1. **常態性**：兩組均通過 Shapiro-Wilk 檢定（p > 0.05）
2. **方差齊性**：通過 Levene 檢定
3. **檢定方法**：自動選擇獨立樣本 t-test
4. **效應量**：Cohen's d 約 0.25（小到中效應）

**商業建議：**
- 若 p < 0.05 且效應量具實質意義 → 建議全量上線新版頁面
- 若效應量過小（d < 0.1） → 需評估工程成本是否值得
- 建議持續監控護欄指標（跳出率、客訴率）
- 若信賴區間跨越零 → 結果不確定，建議延長實驗或增加樣本

---
## Part F — A/A Test（分流系統驗證）

### 為什麼需要 A/A Test？

A/A Test 是在正式 A/B Test 之前，將兩組使用者暴露在**完全相同**的版本下，確認：
1. 分流系統是否正確運作（兩組特徵分布相同）
2. 統計檢定的 Type I Error 是否符合預期（約 5%）
3. 資料收集管道是否有 bug

**預期結果**：p > 0.05（兩組無顯著差異）

In [ ]:
# F-3 A/A Test 模擬
np.random.seed(123)

# 兩組來自完全相同的分布
aa_group1 = np.random.normal(500, 150, 200)
aa_group2 = np.random.normal(500, 150, 200)

# 使用 choose_test 決策函式
aa_result = choose_test([aa_group1, aa_group2])

print("=== A/A Test ===")
for r in aa_result['reasons']:
    print(f"  {r}")
print(f"\np-value: {aa_result['p_value']:.4f}")
print(f"顯著: {'是 — 分流系統可能有問題!' if aa_result['significant'] else '否 — 分流系統正常'}")

In [ ]:
# F-4 模擬多次 A/A Test 以驗證 Type I Error Rate
np.random.seed(456)
n_simulations = 1000
false_positives = 0
p_values_aa = []

for _ in range(n_simulations):
    g1 = np.random.normal(500, 150, 200)
    g2 = np.random.normal(500, 150, 200)
    _, p = stats.ttest_ind(g1, g2)
    p_values_aa.append(p)
    if p < 0.05:
        false_positives += 1

fpr = false_positives / n_simulations
print(f"模擬 {n_simulations} 次 A/A Test")
print(f"False Positive Rate: {fpr:.4f} (預期約 0.05)")
print(f"結論: {'正常' if 0.03 < fpr < 0.07 else '異常'} — FPR 在合理範圍內")

In [ ]:
# F-5 A/A Test p-value 分布應為均勻分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方圖
axes[0].hist(p_values_aa, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axhline(y=n_simulations/20, color='red', linestyle='--', label='Expected (uniform)')
axes[0].set_xlabel('p-value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('A/A Test: p-value Distribution (should be uniform)')
axes[0].legend()

# QQ plot vs uniform
sorted_p = np.sort(p_values_aa)
expected_p = np.linspace(0, 1, n_simulations)
axes[1].scatter(expected_p, sorted_p, s=5, alpha=0.5)
axes[1].plot([0, 1], [0, 1], 'r--', label='Perfect uniform')
axes[1].set_xlabel('Expected (Uniform)')
axes[1].set_ylabel('Observed p-values')
axes[1].set_title('A/A Test: QQ Plot of p-values')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part G — 多重檢定校正

### G-1 為什麼需要多重檢定校正？

當同時進行 k 次檢定時，至少出現一次 Type I Error 的機率：

$$P(\text{至少一個假陽性}) = 1 - (1 - \alpha)^k$$

| 檢定次數 k | 族系錯誤率 (FWER) |
|------------|------------------|
| 1 | 5.0% |
| 5 | 22.6% |
| 10 | 40.1% |
| 20 | 64.2% |
| 50 | 92.3% |

**常見校正方法：**

| 方法 | 控制目標 | 嚴格度 | 適用場景 |
|------|---------|--------|----------|
| Bonferroni | FWER | 最嚴格 | 少量檢定、不允許假陽性 |
| Holm-Bonferroni | FWER | 較嚴格 | Bonferroni 的改良版 |
| Benjamini-Hochberg | FDR | 適中 | 大量檢定、探索性分析 |
| Bonferroni-Holm | FWER | 較嚴格 | 逐步校正 |

In [ ]:
# G-2 模擬族系錯誤率膨脹
np.random.seed(999)

k_values = [1, 2, 5, 10, 20, 50]
theoretical_fwer = [1 - (1 - 0.05)**k for k in k_values]

# 實際模擬
n_sim = 5000
empirical_fwer = []

for k in k_values:
    any_sig_count = 0
    for _ in range(n_sim):
        # 所有 k 個檢定都是 H0 為真（無差異）
        pvals = [stats.ttest_ind(
            np.random.normal(0, 1, 100),
            np.random.normal(0, 1, 100)
        )[1] for _ in range(k)]
        if min(pvals) < 0.05:
            any_sig_count += 1
    empirical_fwer.append(any_sig_count / n_sim)

print("=== 族系錯誤率 (FWER) 膨脹 ===")
print(f"{'k':>4s}  {'理論值':>8s}  {'模擬值':>8s}")
for k, t, e in zip(k_values, theoretical_fwer, empirical_fwer):
    print(f"{k:4d}  {t:8.4f}  {e:8.4f}")

In [ ]:
from statsmodels.stats.multitest import multipletests

# G-3 模擬多重檢定問題
np.random.seed(789)

# 10 個指標的 A/B Test：前 3 個有真實差異，後 7 個無差異
n_tests = 10
p_values = []
labels = []

for i in range(n_tests):
    if i < 3:  # 有真實差異
        g1 = np.random.normal(500, 100, 200)
        g2 = np.random.normal(540, 100, 200)
        labels.append(f'Metric_{i+1} (真實差異)')
    else:       # 無差異
        g1 = np.random.normal(500, 100, 200)
        g2 = np.random.normal(500, 100, 200)
        labels.append(f'Metric_{i+1} (無差異)')

    _, p = stats.ttest_ind(g1, g2)
    p_values.append(p)

print("=== 原始 p-values ===")
for label, p in zip(labels, p_values):
    sig = ' *' if p < 0.05 else ''
    print(f"  {label}: p={p:.4f}{sig}")

In [ ]:
# G-4 套用不同校正方法比較
methods = {
    'Bonferroni': 'bonferroni',
    'Holm-Bonferroni': 'holm',
    'Benjamini-Hochberg (FDR)': 'fdr_bh',
}

correction_results = pd.DataFrame({'Metric': labels, 'p_original': p_values})
correction_results['sig_original'] = correction_results['p_original'] < 0.05

for name, method in methods.items():
    reject, p_corrected, _, _ = multipletests(
        p_values, alpha=0.05, method=method
    )
    correction_results[f'p_{name}'] = p_corrected
    correction_results[f'sig_{name}'] = reject

print("=== 多重檢定校正比較 ===")
display_cols = ['Metric', 'p_original', 'sig_original']
for name in methods:
    display_cols.extend([f'p_{name}', f'sig_{name}'])

correction_results[display_cols].round(4)

In [ ]:
# G-5 視覺化校正效果
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(n_tests)
width = 0.2

ax.bar(x - 1.5*width, correction_results['p_original'], width, label='Original', color='steelblue')
ax.bar(x - 0.5*width, correction_results['p_Bonferroni'], width, label='Bonferroni', color='coral')
ax.bar(x + 0.5*width, correction_results['p_Holm-Bonferroni'], width, label='Holm', color='mediumseagreen')
ax.bar(x + 1.5*width, correction_results['p_Benjamini-Hochberg (FDR)'], width, label='BH (FDR)', color='mediumpurple')

ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5, label='alpha=0.05')
ax.set_xlabel('Metric')
ax.set_ylabel('p-value')
ax.set_title('Multiple Testing Correction Comparison')
ax.set_xticks(x)
ax.set_xticklabels([f'M{i+1}' for i in range(n_tests)], rotation=0)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# 總結
print("\n=== 各方法顯著指標數量 ===")
print(f"原始 (未校正):    {correction_results['sig_original'].sum()} / {n_tests}")
for name in methods:
    print(f"{name:25s}: {correction_results[f'sig_{name}'].sum()} / {n_tests}")
print(f"\n真實有差異的指標: 3 / {n_tests}")

### G-6 校正方法選擇指南

| 情境 | 建議方法 | 理由 |
|------|---------|------|
| 臨床試驗（不容許假陽性） | Bonferroni | 最嚴格，控制 FWER |
| 基因體學（大量檢定） | BH (FDR) | 平衡探索力與假陽性 |
| A/B Test 多指標 | Holm-Bonferroni | 比 Bonferroni 稍寬鬆但仍控制 FWER |
| 探索性資料分析 | BH (FDR) 或不校正（但需標註） | 避免過度保守 |

---
## Part H — 課程總結

### H-1 推論統計 13 章回顧

| 章節 | 主題 | 核心概念 |
|------|------|----------|
| Ch 1 | 假設檢定基礎 | H0/H1、p-value、Type I/II Error |
| Ch 2 | 卡方檢定 | 獨立性、適合度、列聯表 |
| Ch 3 | ANOVA | 組間/組內變異、F 統計量、事後比較 |
| Ch 4 | A/B Testing 流程 | 實驗設計、分流、決策 |
| Ch 5 | 信賴區間 | 區間估計、誤差邊界 |
| Ch 6 | 效應量 | Cohen's d、r、eta-squared |
| Ch 7 | 統計檢力 | Power Analysis、樣本數計算 |
| Ch 8 | 無母數檢定 | Mann-Whitney、Kruskal-Wallis |
| Ch 9 | 相關分析 | Pearson、Spearman、偏相關 |
| Ch 10 | 迴歸基礎 | 簡單/多元迴歸、R-squared |
| Ch 11 | 模型診斷 | 殘差分析、多重共線性 |
| Ch 12 | 貝氏統計 | 先驗/後驗、貝氏因子 |
| **Ch 13** | **決策樹與 A/B 實戰** | **自動化框架、多重校正** |

### H-2 關鍵學習成果

本章你學會了：

1. **統計檢定決策樹**
   - 根據資料類型、組數、常態性、方差齊性選擇正確的檢定方法
   - 使用自動化決策函式 `choose_test()`

2. **自動化檢定框架**
   - `UnivariateAnalysisModule` 一次完成常態性/方差/檢定/效應量/校正
   - `create_codebook()` 快速了解資料概況

3. **A/B Testing 九步流程**
   - 從假設建立到統計分析到商業決策
   - Power Analysis 事前計算樣本數

4. **A/A Test**
   - 驗證分流系統的正確性
   - 確認 Type I Error Rate 符合預期

5. **多重檢定校正**
   - Bonferroni（嚴格控制 FWER）
   - Benjamini-Hochberg（控制 FDR，適合探索性分析）

### H-3 實務工作流速查

```
收到需求
  │
  ├─ 能做實驗嗎？ ──Yes──→ A/B Test 九步流程
  │                  │
  │                  No
  │                  │
  │                  ▼
  │          觀察性研究 + 因果推論方法
  │
  ├─ 幾個指標？ ──1──→ 單一檢定
  │              │
  │             2+──→ 多重校正
  │
  └─ 報告結果
       ├── p-value
       ├── 效應量 + CI
       └── 商業建議
```

### H-4 下一步建議

| 方向 | 建議主題 | 工具/套件 |
|------|---------|----------|
| 進階實驗 | Multi-Armed Bandit | `scipy`, 自建 |
| 進階實驗 | Sequential Testing (CUSUM) | `statsmodels` |
| 因果推論 | Propensity Score Matching | `causalinference` |
| 因果推論 | Difference-in-Differences | `linearmodels` |
| 貝氏方法 | Bayesian A/B Testing | `pymc`, `arviz` |
| 工程實踐 | 分流系統設計 | Feature Flag 平台 |
| 工程實踐 | 實驗平台架構 | Experimentation Platform |

### H-5 練習題

#### 練習 1：決策樹應用
- 載入 Titanic 資料集
- 使用 `choose_test()` 函式比較男性與女性的 Fare 分布
- 記錄自動選擇的檢定方法和結果

#### 練習 2：A/B Test 設計
- 設定情境：電商 App 改版後的「加入購物車」點擊率
- 進行 Power Analysis 計算所需樣本數
- 模擬資料並完成完整的九步流程

#### 練習 3：多重校正
- 模擬 20 個指標的 A/B Test（其中 5 個有真實差異）
- 比較 Bonferroni、Holm、BH 三種校正方法
- 計算各方法的 True Positive Rate 和 False Positive Rate

In [ ]:
# TODO: 練習 1 — 使用 choose_test() 比較男性與女性的 Fare
# male_fare = df_titanic[df_titanic['Sex'] == 'male']['Fare'].dropna().values
# female_fare = df_titanic[df_titanic['Sex'] == 'female']['Fare'].dropna().values
# result = choose_test([male_fare, female_fare])
# for r in result['reasons']:
#     print(r)
# print(f"Test: {result['test']}, p={result['p_value']:.4f}")

In [ ]:
# TODO: 練習 2 — 完整 A/B Test 九步流程
# 提示：
# 1. 設定 H0/H1
# 2. 定義主指標（點擊率）
# 3. Power Analysis (使用 proportion_effectsize + zt_ind_solve_power)
# 4. 模擬分流
# 5. A/A Test
# 6-7. 模擬實驗資料 (np.random.binomial)
# 8. 統計分析 (proportions_ztest)
# 9. 撰寫結論

In [ ]:
# TODO: 練習 3 — 多重校正比較
# 提示：
# 1. 模擬 20 組 A/B Test
# 2. 前 5 組設定 treatment_mean = control_mean + delta
# 3. 後 15 組設定 treatment_mean = control_mean
# 4. 使用 multipletests() 套用三種校正
# 5. 計算 TPR = TP / (TP + FN), FPR = FP / (FP + TN)

---

| 導覽 | 連結 |
|------|------|
| 上一章 | 12_迴歸分析與模型診斷 |
| 下一章 | 14_進階主題與專案整合 |

---
*本教材為 iSpan 資料分析課程系列*